
# Starts-at-Zero Checker — Clean Hybrid OCR Version

This notebook uses a safer hybrid OCR design:

1. **EasyOCR first**: detect and read numeric axis labels.
2. **Scoring system**: choose the most likely value axis from EasyOCR only.
3. **Tesseract fallback**: only runs if EasyOCR cannot prove that the axis starts at zero.
4. **Final rescue**: small targeted crop around the expected zero area.

Important design rule: **Tesseract is not merged into the main OCR list before scoring.** This prevents Tesseract noise from influencing the first axis decision.


In [ ]:

# Install only once if your environment does not already have these packages.
# On macOS, pytesseract also needs the real Tesseract binary:
#   brew install tesseract

# %pip install opencv-python easyocr pytesseract pillow pandas matplotlib numpy


In [1]:

import os
import re
from shutil import which
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import easyocr
import pytesseract
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage


In [2]:

# -----------------------------
# Configuration
# -----------------------------

OCR_SCALE = 2.0
EASYOCR_MIN_CONF = 0.20

# Keep Tesseract strict. Do not use PSM 11/13 for this task.
TESSERACT_MIN_CONF = 20.0
TESSERACT_PSM_MODES = (4, 6)
TESSERACT_WHITELIST = "0123456789.,%$€£+-Oo"

VERTICAL_LEFT_STRIP_RATIO = 0.18       # Used only when EasyOCR cannot form an axis.
HORIZONTAL_BOTTOM_STRIP_RATIO = 0.25   # Used only when EasyOCR cannot form an axis.

ZERO_ABS_TOLERANCE = 1e-9

# Setup Tesseract command path.
# You may override it manually if needed:
# pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"
tesseract_path = os.getenv("TESSERACT_CMD") or which("tesseract") or "/opt/homebrew/bin/tesseract"
pytesseract.pytesseract.tesseract_cmd = tesseract_path
print("Tesseract path:", pytesseract.pytesseract.tesseract_cmd)


Tesseract path: /opt/homebrew/bin/tesseract


In [3]:

# Initialize EasyOCR once. This may take a moment.
reader = easyocr.Reader(['en'], gpu=False)


Using CPU. Note: This module is much faster with a GPU.


In [4]:

# -----------------------------
# Text parsing and box helpers
# -----------------------------

def normalize_ocr_text(text):
    """Normalize common OCR confusions without being too aggressive."""
    if text is None:
        return ""
    text = str(text).strip()
    text = text.replace("O", "0").replace("o", "0")
    text = text.replace("−", "-").replace("–", "-")
    return text


def extract_number(text):
    """Extract the first numeric value from OCR text.

    Examples:
    - "0%" -> 0.0
    - "20%" -> 20.0
    - "$1,200" -> 1200.0
    - "O" -> 0.0
    """
    text = normalize_ocr_text(text)
    text = text.replace(",", "")
    text = text.replace("%", "")
    text = text.replace("$", "").replace("€", "").replace("£", "")

    match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
    if not match:
        return None
    try:
        return float(match.group(0))
    except ValueError:
        return None


def is_zero_value(value):
    return value is not None and abs(float(value)) <= ZERO_ABS_TOLERANCE


def easyocr_box_geometry(box, scale=OCR_SCALE):
    """Convert EasyOCR box from resized-image coordinates back to original-image coordinates."""
    xs = [p[0] / scale for p in box]
    ys = [p[1] / scale for p in box]
    return {
        "left": float(min(xs)),
        "right": float(max(xs)),
        "top": float(min(ys)),
        "bottom": float(max(ys)),
        "cx": float(sum(xs) / 4),
        "cy": float(sum(ys) / 4),
    }


def make_detection(value, text, confidence, source, geo, stage):
    item = {
        "value": float(value),
        "text": str(text),
        "confidence": float(confidence),
        "source": source,
        "stage": stage,
    }
    item.update({k: float(v) for k, v in geo.items()})
    return item


def dedupe_detections(detections, center_tol=8, value_tol=1e-6):
    """Remove near-duplicate OCR detections, keeping the highest confidence one."""
    if not detections:
        return []

    detections = sorted(detections, key=lambda d: d.get("confidence", 0), reverse=True)
    kept = []

    for det in detections:
        duplicate = False
        for old in kept:
            same_value = abs(det["value"] - old["value"]) <= value_tol
            close_center = abs(det["cx"] - old["cx"]) <= center_tol and abs(det["cy"] - old["cy"]) <= center_tol
            if same_value and close_center:
                duplicate = True
                break
        if not duplicate:
            kept.append(det)

    return kept


In [5]:

# -----------------------------
# EasyOCR main pass
# -----------------------------

def preprocess_for_easyocr(image):
    """Preprocess the full chart for EasyOCR.

    EasyOCR runs on a 2x image, but all boxes are converted back to original coordinates.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=OCR_SCALE, fy=OCR_SCALE, interpolation=cv2.INTER_CUBIC)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return gray


def ocr_numeric_boxes_easyocr(image):
    """Run EasyOCR and return numeric detections in original image coordinates."""
    processed = preprocess_for_easyocr(image)
    results = reader.readtext(processed)

    numeric_boxes = []
    for box, text, confidence in results:
        if confidence < EASYOCR_MIN_CONF:
            continue

        value = extract_number(text)
        if value is None:
            continue

        geo = easyocr_box_geometry(box, scale=OCR_SCALE)
        numeric_boxes.append(
            make_detection(
                value=value,
                text=text,
                confidence=confidence,
                source="easyocr",
                geo=geo,
                stage="easyocr_main",
            )
        )

    return dedupe_detections(numeric_boxes)


In [6]:

# -----------------------------
# Axis scoring system
# -----------------------------

def is_monotonic(values, allow_equal=False):
    if len(values) < 2:
        return False
    if allow_equal:
        inc = all(values[i] <= values[i + 1] for i in range(len(values) - 1))
        dec = all(values[i] >= values[i + 1] for i in range(len(values) - 1))
    else:
        inc = all(values[i] < values[i + 1] for i in range(len(values) - 1))
        dec = all(values[i] > values[i + 1] for i in range(len(values) - 1))
    return inc or dec


def is_evenly_spaced_values(values, tolerance_ratio=0.25):
    if len(values) < 3:
        return False

    vals = sorted(set(float(v) for v in values))
    if len(vals) < 3:
        return False

    diffs = [vals[i + 1] - vals[i] for i in range(len(vals) - 1)]
    if any(abs(d) <= 1e-9 for d in diffs):
        return False

    avg = sum(diffs) / len(diffs)
    return all(abs(d - avg) <= abs(avg) * tolerance_ratio for d in diffs)


def remove_same_position_duplicates(group, orientation):
    """Keep one label when two detections overlap on the same tick position."""
    if not group:
        return []

    if orientation == "vertical":
        key_name = "cy"
        pos_tol = 10
    else:
        key_name = "cx"
        pos_tol = 12

    group = sorted(group, key=lambda d: d.get("confidence", 0), reverse=True)
    kept = []
    for item in group:
        if not any(abs(item[key_name] - old[key_name]) <= pos_tol for old in kept):
            kept.append(item)
    return kept


def score_vertical_axis_group(group, image_width, image_height):
    group = remove_same_position_duplicates(group, orientation="vertical")
    if len(group) < 3:
        return -1, group

    group = sorted(group, key=lambda d: d["cy"])
    values_by_y = [d["value"] for d in group]

    mean_right = np.mean([d["right"] for d in group])
    std_right = np.std([d["right"] for d in group])
    y_spread = max(d["cy"] for d in group) - min(d["cy"] for d in group)

    score = 0

    # Axis labels should usually be on the left side for vertical charts.
    if mean_right < image_width * 0.35:
        score += 4
    if mean_right < image_width * 0.22:
        score += 2

    # Right edges should be aligned.
    if std_right < image_width * 0.025:
        score += 3
    elif std_right < image_width * 0.05:
        score += 1

    # Value labels should span a meaningful vertical range.
    if y_spread > image_height * 0.30:
        score += 3
    if y_spread > image_height * 0.45:
        score += 1

    # Values should be monotonic from top to bottom.
    if is_monotonic(values_by_y):
        score += 4

    # Tick values often have regular spacing.
    if is_evenly_spaced_values(values_by_y):
        score += 3

    # More labels are usually better, but cap the reward.
    score += min(len(group), 6)

    return score, group


def find_vertical_value_axis(numbers, image_width, image_height, return_debug=False):
    if len(numbers) < 3:
        return ([], []) if return_debug else []

    bin_size = max(image_width * 0.04, 20)
    groups = defaultdict(list)

    # Y-axis labels are normally right-aligned, so group by right edge.
    for item in numbers:
        x_bin = int(item["right"] // bin_size)
        groups[x_bin].append(item)

    best_group = []
    best_score = -1
    debug_rows = []

    for x_bin, group in groups.items():
        score, cleaned = score_vertical_axis_group(group, image_width, image_height)
        debug_rows.append({
            "bin": x_bin,
            "score": score,
            "count": len(cleaned),
            "values": [g["value"] for g in sorted(cleaned, key=lambda d: d["cy"])],
            "sources": sorted(set(g["source"] for g in cleaned)),
        })
        if score > best_score:
            best_score = score
            best_group = cleaned

    # Require a minimum score to avoid accepting random data labels.
    if best_score < 10:
        best_group = []

    if return_debug:
        return best_group, debug_rows
    return best_group


def score_horizontal_axis_group(group, image_width, image_height):
    group = remove_same_position_duplicates(group, orientation="horizontal")
    if len(group) < 3:
        return -1, group

    group = sorted(group, key=lambda d: d["cx"])
    values_by_x = [d["value"] for d in group]

    mean_cy = np.mean([d["cy"] for d in group])
    std_cy = np.std([d["cy"] for d in group])
    x_spread = max(d["cx"] for d in group) - min(d["cx"] for d in group)

    score = 0

    # X-axis labels should usually be in the lower part of horizontal charts.
    if mean_cy > image_height * 0.55:
        score += 4
    if mean_cy > image_height * 0.70:
        score += 2

    # Tick labels should be horizontally aligned.
    if std_cy < image_height * 0.03:
        score += 3
    elif std_cy < image_height * 0.06:
        score += 1

    if x_spread > image_width * 0.30:
        score += 3
    if x_spread > image_width * 0.45:
        score += 1

    if is_monotonic(values_by_x):
        score += 4

    if is_evenly_spaced_values(values_by_x):
        score += 3

    score += min(len(group), 6)
    return score, group


def find_horizontal_value_axis(numbers, image_width, image_height, return_debug=False):
    if len(numbers) < 3:
        return ([], []) if return_debug else []

    bin_size = max(image_height * 0.06, 20)
    groups = defaultdict(list)

    # X-axis labels are normally horizontally aligned, so group by center y.
    for item in numbers:
        y_bin = int(item["cy"] // bin_size)
        groups[y_bin].append(item)

    best_group = []
    best_score = -1
    debug_rows = []

    for y_bin, group in groups.items():
        score, cleaned = score_horizontal_axis_group(group, image_width, image_height)
        debug_rows.append({
            "bin": y_bin,
            "score": score,
            "count": len(cleaned),
            "values": [g["value"] for g in sorted(cleaned, key=lambda d: d["cx"])],
            "sources": sorted(set(g["source"] for g in cleaned)),
        })
        if score > best_score:
            best_score = score
            best_group = cleaned

    if best_score < 10:
        best_group = []

    if return_debug:
        return best_group, debug_rows
    return best_group


def find_value_axis(numbers, image_width, image_height, orientation, return_debug=False):
    if orientation == "vertical":
        return find_vertical_value_axis(numbers, image_width, image_height, return_debug=return_debug)
    if orientation == "horizontal":
        return find_horizontal_value_axis(numbers, image_width, image_height, return_debug=return_debug)
    raise ValueError("orientation must be 'vertical' or 'horizontal'")


def axis_contains_zero(axis_group):
    return any(is_zero_value(item["value"]) for item in axis_group)


In [7]:

# -----------------------------
# Tesseract helpers
# -----------------------------

def preprocess_variants_for_tesseract(crop, scale=3.0):
    """Return a small number of strict preprocessing variants.

    Do not add too many variants. More variants increase false positives.
    """
    if crop.size == 0:
        return []

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop.copy()
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    variants = []
    variants.append(("gray", gray))

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(gray)
    variants.append(("clahe", clahe_img))

    # One threshold variant. Avoid adaptive threshold here because it often turns gridlines into fake text.
    _, otsu = cv2.threshold(clahe_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(("otsu", otsu))

    return variants


def run_tesseract_numeric_on_crop(
    crop,
    crop_left=0,
    crop_top=0,
    scale=3.0,
    psm_modes=TESSERACT_PSM_MODES,
    min_conf=TESSERACT_MIN_CONF,
    stage="tesseract",
):
    """Run Tesseract on a crop and map boxes back to original image coordinates."""
    detections = []

    for variant_name, img in preprocess_variants_for_tesseract(crop, scale=scale):
        for psm in psm_modes:
            config = f"--oem 3 --psm {psm} -c tessedit_char_whitelist={TESSERACT_WHITELIST}"
            try:
                data = pytesseract.image_to_data(
                    img,
                    config=config,
                    output_type=pytesseract.Output.DICT,
                )
            except Exception as exc:
                print("Tesseract failed:", exc)
                continue

            n = len(data.get("text", []))
            for i in range(n):
                text = data["text"][i]
                value = extract_number(text)
                if value is None:
                    continue

                try:
                    conf = float(data["conf"][i])
                except Exception:
                    conf = -1.0

                if conf < min_conf:
                    continue

                x = float(data["left"][i]) / scale + crop_left
                y = float(data["top"][i]) / scale + crop_top
                w_box = float(data["width"][i]) / scale
                h_box = float(data["height"][i]) / scale

                if w_box <= 1 or h_box <= 1:
                    continue

                geo = {
                    "left": x,
                    "right": x + w_box,
                    "top": y,
                    "bottom": y + h_box,
                    "cx": x + w_box / 2,
                    "cy": y + h_box / 2,
                }
                detections.append(
                    make_detection(
                        value=value,
                        text=text,
                        confidence=conf,
                        source="tesseract",
                        geo=geo,
                        stage=f"{stage}:{variant_name}:psm{psm}",
                    )
                )

    return dedupe_detections(detections, center_tol=7)


In [8]:

# -----------------------------
# Tesseract fallback logic
# -----------------------------

def estimate_zero_position_from_axis(axis_group, orientation, image_width, image_height):
    """Estimate where zero should be based on detected axis labels.

    Returns a crop rectangle (x1, y1, x2, y2) in original image coordinates.
    """
    if len(axis_group) < 2:
        return None

    values = np.array([d["value"] for d in axis_group], dtype=float)

    if orientation == "vertical":
        positions = np.array([d["cy"] for d in axis_group], dtype=float)
        # cy = a * value + b
        if len(set(values)) < 2:
            return None
        a, b = np.polyfit(values, positions, 1)
        y0 = float(a * 0 + b)

        mean_left = float(np.mean([d["left"] for d in axis_group]))
        mean_right = float(np.mean([d["right"] for d in axis_group]))
        label_width = max(25.0, mean_right - mean_left)

        x1 = max(0, int(mean_left - label_width * 1.5 - image_width * 0.02))
        x2 = min(image_width, int(mean_right + label_width * 1.2 + image_width * 0.02))
        y1 = max(0, int(y0 - image_height * 0.06))
        y2 = min(image_height, int(y0 + image_height * 0.06))

        # Reject impossible predicted locations.
        if y2 <= 0 or y1 >= image_height or x2 <= x1 or y2 <= y1:
            return None
        return x1, y1, x2, y2

    if orientation == "horizontal":
        positions = np.array([d["cx"] for d in axis_group], dtype=float)
        if len(set(values)) < 2:
            return None
        a, b = np.polyfit(values, positions, 1)
        x0 = float(a * 0 + b)

        mean_top = float(np.mean([d["top"] for d in axis_group]))
        mean_bottom = float(np.mean([d["bottom"] for d in axis_group]))
        label_height = max(18.0, mean_bottom - mean_top)

        x1 = max(0, int(x0 - image_width * 0.06))
        x2 = min(image_width, int(x0 + image_width * 0.06))
        y1 = max(0, int(mean_top - label_height * 1.5 - image_height * 0.02))
        y2 = min(image_height, int(mean_bottom + label_height * 1.2 + image_height * 0.02))

        if x2 <= 0 or x1 >= image_width or x2 <= x1 or y2 <= y1:
            return None
        return x1, y1, x2, y2

    raise ValueError("orientation must be 'vertical' or 'horizontal'")


def tesseract_zero_near_expected_position(image, axis_group, orientation):
    """Targeted Tesseract fallback.

    This is used only after EasyOCR axis scoring has run and zero is missing.
    It searches a small crop where zero is expected, not the whole chart.
    """
    h, w = image.shape[:2]
    rect = estimate_zero_position_from_axis(axis_group, orientation, w, h)
    if rect is None:
        return []

    x1, y1, x2, y2 = rect
    crop = image[y1:y2, x1:x2]

    detections = run_tesseract_numeric_on_crop(
        crop,
        crop_left=x1,
        crop_top=y1,
        scale=3.0,
        psm_modes=(6, 7),
        min_conf=12.0,  # slightly lower because the search area is already very constrained
        stage="tesseract_expected_zero",
    )

    zero_detections = [d for d in detections if is_zero_value(d["value"])]
    return zero_detections


def tesseract_axis_strip_rescue(image, orientation):
    """Tesseract fallback if EasyOCR cannot form a valid axis at all.

    This is still restricted to the likely axis strip. It is not used before EasyOCR scoring.
    """
    h, w = image.shape[:2]

    if orientation == "vertical":
        x1, y1 = 0, 0
        x2, y2 = int(w * VERTICAL_LEFT_STRIP_RATIO), h
    elif orientation == "horizontal":
        x1, y1 = 0, int(h * (1.0 - HORIZONTAL_BOTTOM_STRIP_RATIO))
        x2, y2 = w, h
    else:
        raise ValueError("orientation must be 'vertical' or 'horizontal'")

    crop = image[y1:y2, x1:x2]
    detections = run_tesseract_numeric_on_crop(
        crop,
        crop_left=x1,
        crop_top=y1,
        scale=3.0,
        psm_modes=(4, 6),
        min_conf=25.0,
        stage="tesseract_axis_strip_rescue",
    )
    return detections


In [9]:

# -----------------------------
# Final EasyOCR rescue
# -----------------------------

def easyocr_zero_rescue_crop(image, orientation):
    """Last rescue pass using EasyOCR on a small likely-zero area.

    This is intentionally last. It cannot affect the initial scoring system.
    """
    h, w = image.shape[:2]

    if orientation == "vertical":
        x1, y1 = 0, int(h * 0.65)
        x2, y2 = int(w * 0.28), h
    elif orientation == "horizontal":
        x1, y1 = 0, int(h * 0.65)
        x2, y2 = int(w * 0.35), h
    else:
        raise ValueError("orientation must be 'vertical' or 'horizontal'")

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        return []

    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    try:
        results = reader.readtext(gray, allowlist="0123456789.%Oo")
    except TypeError:
        results = reader.readtext(gray)

    detections = []
    scale = 3.0
    for box, text, confidence in results:
        if confidence < 0.10:
            continue
        value = extract_number(text)
        if not is_zero_value(value):
            continue

        xs = [p[0] / scale + x1 for p in box]
        ys = [p[1] / scale + y1 for p in box]
        geo = {
            "left": min(xs),
            "right": max(xs),
            "top": min(ys),
            "bottom": max(ys),
            "cx": sum(xs) / 4,
            "cy": sum(ys) / 4,
        }
        detections.append(
            make_detection(
                value=value,
                text=text,
                confidence=confidence,
                source="easyocr",
                geo=geo,
                stage="easyocr_final_zero_rescue",
            )
        )

    return dedupe_detections(detections)


In [10]:

# -----------------------------
# Final decision function
# -----------------------------

def summarize_axis_group(axis_group):
    if not axis_group:
        return []
    key = "cy" if np.std([d["cy"] for d in axis_group]) > np.std([d["cx"] for d in axis_group]) else "cx"
    return [
        {
            "value": d["value"],
            "text": d["text"],
            "source": d["source"],
            "confidence": round(d["confidence"], 3),
            "cx": round(d["cx"], 1),
            "cy": round(d["cy"], 1),
        }
        for d in sorted(axis_group, key=lambda x: x[key])
    ]


def check_starts_at_zero(image_path, orientation="vertical", debug=False):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found: {image_path}")

    h, w = image.shape[:2]
    trace = []

    # 1. EasyOCR main pass.
    easy_numbers = ocr_numeric_boxes_easyocr(image)
    axis_group, score_debug = find_value_axis(easy_numbers, w, h, orientation, return_debug=True)
    trace.append(f"EasyOCR numeric detections: {len(easy_numbers)}")
    trace.append(f"EasyOCR selected axis labels: {len(axis_group)}")

    if axis_contains_zero(axis_group):
        result = {
            "status": "compliant",
            "starts_at_zero": True,
            "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
            "zero_source": "easyocr_axis_group",
            "axis_values": [d["value"] for d in axis_group],
            "axis_group": summarize_axis_group(axis_group),
            "trace": trace,
        }
        if debug:
            result["easyocr_detections"] = easy_numbers
            result["score_debug"] = score_debug
        return result

    # 2. Tesseract expected-zero fallback if EasyOCR found an axis but missed zero.
    tesseract_zero = []
    if axis_group:
        tesseract_zero = tesseract_zero_near_expected_position(image, axis_group, orientation)
        trace.append(f"Tesseract expected-zero detections: {len(tesseract_zero)}")
        if tesseract_zero:
            result = {
                "status": "compliant",
                "starts_at_zero": True,
                "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
                "zero_source": "tesseract_expected_zero_rescue",
                "axis_values": [d["value"] for d in axis_group] + [0.0],
                "axis_group": summarize_axis_group(axis_group),
                "rescued_zero": summarize_axis_group(tesseract_zero),
                "trace": trace,
            }
            if debug:
                result["easyocr_detections"] = easy_numbers
                result["tesseract_detections"] = tesseract_zero
                result["score_debug"] = score_debug
            return result

    # 3. Tesseract axis-strip rescue only if EasyOCR could not form a usable axis.
    t_axis_numbers = []
    t_axis_group = []
    if not axis_group:
        t_axis_numbers = tesseract_axis_strip_rescue(image, orientation)
        t_axis_group, t_score_debug = find_value_axis(t_axis_numbers, w, h, orientation, return_debug=True)
        trace.append(f"Tesseract axis-strip detections: {len(t_axis_numbers)}")
        trace.append(f"Tesseract selected axis labels: {len(t_axis_group)}")

        if axis_contains_zero(t_axis_group):
            result = {
                "status": "compliant",
                "starts_at_zero": True,
                "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
                "zero_source": "tesseract_axis_strip_rescue",
                "axis_values": [d["value"] for d in t_axis_group],
                "axis_group": summarize_axis_group(t_axis_group),
                "trace": trace,
            }
            if debug:
                result["easyocr_detections"] = easy_numbers
                result["tesseract_detections"] = t_axis_numbers
                result["score_debug"] = score_debug
                result["tesseract_score_debug"] = t_score_debug
            return result
    else:
        t_score_debug = []

    # 4. Final small-crop EasyOCR zero rescue.
    final_zero = easyocr_zero_rescue_crop(image, orientation)
    trace.append(f"Final EasyOCR zero rescue detections: {len(final_zero)}")
    if final_zero and axis_group:
        result = {
            "status": "compliant",
            "starts_at_zero": True,
            "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
            "zero_source": "easyocr_final_zero_rescue",
            "axis_values": [d["value"] for d in axis_group] + [0.0],
            "axis_group": summarize_axis_group(axis_group),
            "rescued_zero": summarize_axis_group(final_zero),
            "trace": trace,
        }
        if debug:
            result["easyocr_detections"] = easy_numbers
            result["tesseract_detections"] = tesseract_zero + t_axis_numbers
            result["final_zero_detections"] = final_zero
            result["score_debug"] = score_debug
        return result

    # 5. No zero found.
    if not axis_group and not t_axis_group:
        result = {
            "status": "unknown",
            "starts_at_zero": None,
            "reason": "No reliable value axis was found.",
            "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
            "zero_source": None,
            "axis_values": [],
            "trace": trace,
        }
    else:
        final_axis_group = axis_group if axis_group else t_axis_group
        result = {
            "status": "non_compliant",
            "starts_at_zero": False,
            "reason": "A value axis was found, but no zero label was detected by EasyOCR, Tesseract fallback, or final rescue.",
            "axis_type": "y-axis" if orientation == "vertical" else "x-axis",
            "zero_source": None,
            "axis_values": [d["value"] for d in final_axis_group],
            "axis_group": summarize_axis_group(final_axis_group),
            "trace": trace,
        }

    if debug:
        result["easyocr_detections"] = easy_numbers
        result["tesseract_detections"] = tesseract_zero + t_axis_numbers
        result["final_zero_detections"] = final_zero
        result["score_debug"] = score_debug
        result["tesseract_score_debug"] = t_score_debug
    return result


In [11]:

# -----------------------------
# Debug helpers
# -----------------------------

def debug_ocr_table(image_path, orientation="vertical"):
    result = check_starts_at_zero(image_path, orientation=orientation, debug=True)

    rows = []
    selected_keys = set()
    for d in result.get("axis_group", []):
        selected_keys.add((round(d["cx"], 1), round(d["cy"], 1), d["value"]))

    all_dets = []
    all_dets.extend(result.get("easyocr_detections", []))
    all_dets.extend(result.get("tesseract_detections", []))
    all_dets.extend(result.get("final_zero_detections", []))

    for d in all_dets:
        rows.append({
            "value": d["value"],
            "text": d["text"],
            "source": d["source"],
            "stage": d["stage"],
            "confidence": round(d["confidence"], 3),
            "left": round(d["left"], 1),
            "right": round(d["right"], 1),
            "top": round(d["top"], 1),
            "bottom": round(d["bottom"], 1),
            "cx": round(d["cx"], 1),
            "cy": round(d["cy"], 1),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        sort_col = "cy" if orientation == "vertical" else "cx"
        df = df.sort_values(["source", sort_col, "cx"], ignore_index=True)

    print("Decision result:")
    print({k: v for k, v in result.items() if k not in ["easyocr_detections", "tesseract_detections", "final_zero_detections", "score_debug", "tesseract_score_debug"]})
    print("\nScoring debug:")
    display(pd.DataFrame(result.get("score_debug", [])))
    if result.get("tesseract_score_debug"):
        print("\nTesseract scoring debug:")
        display(pd.DataFrame(result.get("tesseract_score_debug", [])))

    return df


def draw_ocr_debug(image_path, orientation="vertical", output_path="debug_clean_hybrid_ocr.png"):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found: {image_path}")

    result = check_starts_at_zero(image_path, orientation=orientation, debug=True)
    canvas = image.copy()

    all_dets = []
    all_dets.extend(result.get("easyocr_detections", []))
    all_dets.extend(result.get("tesseract_detections", []))
    all_dets.extend(result.get("final_zero_detections", []))

    axis_centers = set()
    for d in result.get("axis_group", []):
        axis_centers.add((round(d["cx"], 1), round(d["cy"], 1), d["value"]))

    for d in all_dets:
        x1, y1 = int(d["left"]), int(d["top"])
        x2, y2 = int(d["right"]), int(d["bottom"])

        key = (round(d["cx"], 1), round(d["cy"], 1), d["value"])

        if key in axis_centers:
            color = (0, 180, 0)       # selected axis label
            thickness = 2
        elif is_zero_value(d["value"]):
            color = (0, 0, 255)       # zero candidate
            thickness = 2
        elif d["source"] == "tesseract":
            color = (0, 140, 255)     # tesseract fallback
            thickness = 1
        else:
            color = (255, 0, 0)       # easyocr non-axis numeric label
            thickness = 1

        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)
        label = f"{d['value']:g} {d['source']}"
        cv2.putText(canvas, label, (x1, max(15, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    cv2.imwrite(output_path, canvas)
    print("Saved debug image to:", output_path)
    print("Decision:", result["status"], "starts_at_zero=", result["starts_at_zero"], "zero_source=", result.get("zero_source"))
    display(IPImage(filename=output_path))
    return result


In [18]:

# -----------------------------
# Example usage
# -----------------------------

IMAGE_PATH = "/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/test5.png"
ORIENTATION = "horizontal"  # "vertical" or "horizontal"

result = check_starts_at_zero(IMAGE_PATH, orientation=ORIENTATION, debug=True)
print(result)

# df = debug_ocr_table(IMAGE_PATH, orientation=ORIENTATION)
# display(df)

# draw_ocr_debug(IMAGE_PATH, orientation=ORIENTATION)


{'status': 'compliant', 'starts_at_zero': True, 'axis_type': 'x-axis', 'zero_source': 'easyocr_axis_group', 'axis_values': [0.0, 59.0, 88.0], 'axis_group': [{'value': 0.0, 'text': 'Missouri', 'source': 'easyocr', 'confidence': 1.0, 'cx': 188.8, 'cy': 807.5}, {'value': 59.0, 'text': '59', 'source': 'easyocr', 'confidence': 1.0, 'cx': 470.0, 'cy': 810.5}, {'value': 88.0, 'text': '88', 'source': 'easyocr', 'confidence': 1.0, 'cx': 556.5, 'cy': 803.0}], 'trace': ['EasyOCR numeric detections: 33', 'EasyOCR selected axis labels: 3'], 'easyocr_detections': [{'value': 49.0, 'text': '49', 'confidence': 1.0, 'source': 'easyocr', 'stage': 'easyocr_main', 'left': 416.5, 'right': 459.5, 'top': 305.5, 'bottom': 339.0, 'cx': 438.0, 'cy': 322.25}, {'value': 188.0, 'text': '188', 'confidence': 0.9999995870469189, 'source': 'easyocr', 'stage': 'easyocr_main', 'left': 844.5, 'right': 901.5, 'top': 62.5, 'bottom': 93.5, 'cx': 873.0, 'cy': 78.0}, {'value': 125.0, 'text': '125', 'confidence': 0.999999587046